In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Lokasi folder utama project
ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/dataset_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "data/processed/dataset_cleaned.csv").exists(), "Dataset tidak ditemukan"

# Membaca dataset hasil cleaning
df = pd.read_csv(
    ROOT / "data/processed/dataset_cleaned.csv"
)

# Menampilkan 5 baris pertama
display(df.head())

,run_id,timestamp,sesi,putaran,resolver,ip_resolver,domain,jenis_cache,query_name,waktu_ms,status,rtt_ms
0,bfcc6beb3ee9,2026-11-24 04:49:38,malam,1,cloudflare,1.1.1.1,detik.com,acak,ypxjjg10kxkaznh4.detik.com,15.349,nxdomain,6.545
1,bfcc6beb3ee9,2026-11-24 04:49:39,malam,1,cloudflare,1.1.1.1,detik.com,ulang,detik.com,10.293,ok,6.545
2,bfcc6beb3ee9,2026-11-24 04:49:39,malam,1,google,8.8.8.8,detik.com,acak,creg0axmee0cdrg2.detik.com,15.583,nxdomain,6.460
3,bfcc6beb3ee9,2026-11-24 04:49:40,malam,1,google,8.8.8.8,detik.com,ulang,detik.com,9.627,ok,6.460
4,bfcc6beb3ee9,2026-11-24 04:49:40,malam,1,isp,103.29.106.17,detik.com,acak,u0h78qi5s5l62i99.detik.com,17.522,nxdomain,7.444


In [2]:
# Jumlah baris dan kolom
print("Ukuran dataset:", df.shape)

# Informasi tipe data setiap kolom
print("\n=== INFORMASI DATASET ===")
df.info()

# Memeriksa jumlah nilai kosong
print("\n=== MISSING VALUES ===")
display(df.isnull().sum())

# Memeriksa jumlah observasi setiap resolver
print("\n=== JUMLAH PER RESOLVER ===")
display(df["resolver"].value_counts())

Ukuran dataset: (600, 12)

=== INFORMASI DATASET ===
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   run_id       600 non-null    str    
 1   timestamp    600 non-null    str    
 2   sesi         600 non-null    str    
 3   putaran      600 non-null    int64  
 4   resolver     600 non-null    str    
 5   ip_resolver  600 non-null    str    
 6   domain       600 non-null    str    
 7   jenis_cache  600 non-null    str    
 8   query_name   600 non-null    str    
 9   waktu_ms     600 non-null    float64
 10  status       600 non-null    str    
 11  rtt_ms       600 non-null    float64
dtypes: float64(2), int64(1), str(9)
memory usage: 56.4 KB

=== MISSING VALUES ===


run_id         0
timestamp      0
sesi           0
putaran        0
resolver       0
ip_resolver    0
domain         0
jenis_cache    0
query_name     0
waktu_ms       0
status         0
rtt_ms         0
dtype: int64


=== JUMLAH PER RESOLVER ===


resolver
cloudflare    200
google        200
isp           200
Name: count, dtype: int64

In [3]:
# Memilih data analisis utama
utama = df[
    (df["jenis_cache"] == "ulang") &
    (df["status"] == "ok")
].copy()

# Menghitung statistik deskriptif
statistik = utama.groupby("resolver")["waktu_ms"].agg(
    jumlah="count",
    mean="mean",
    median="median",
    minimum="min",
    maksimum="max",
    varians="var",
    standar_deviasi="std",
    Q1=lambda x: x.quantile(0.25),
    Q2=lambda x: x.quantile(0.50),
    Q3=lambda x: x.quantile(0.75)
)

display(statistik.round(3))

,jumlah,mean,median,minimum,maksimum,varians,standar_deviasi,Q1,Q2,Q3
resolver,,,,,,,,,,
cloudflare,100,15.323,10.685,7.031,127.429,351.593,18.751,10.008,10.685,12.021
google,100,15.266,10.566,8.935,93.205,225.555,15.018,9.922,10.566,12.110
isp,100,15.291,10.648,6.975,116.147,301.610,17.367,9.895,10.648,11.856


In [4]:
# Memilih data query acak
acak = df[
    df["jenis_cache"] == "acak"
].copy()

# Statistik deskriptif query acak
statistik_acak = acak.groupby("resolver")["waktu_ms"].agg(
    jumlah="count",
    mean="mean",
    median="median",
    minimum="min",
    maksimum="max",
    varians="var",
    standar_deviasi="std",
    Q1=lambda x: x.quantile(0.25),
    Q2=lambda x: x.quantile(0.50),
    Q3=lambda x: x.quantile(0.75)
)

display(statistik_acak.round(3))

,jumlah,mean,median,minimum,maksimum,varians,standar_deviasi,Q1,Q2,Q3
resolver,,,,,,,,,,
cloudflare,100,28.764,14.112,11.276,289.936,1969.499,44.379,13.228,14.112,16.648
google,100,26.959,14.550,11.936,286.424,1740.301,41.717,13.352,14.550,16.142
isp,100,27.374,14.154,9.947,203.860,1526.718,39.073,13.429,14.154,16.980


In [5]:
# Folder penyimpanan hasil analisis
hasil_dir = ROOT / "data/processed"
hasil_dir.mkdir(parents=True, exist_ok=True)

# Simpan hasil statistik deskriptif
statistik.round(3).to_csv(
    hasil_dir / "statistik_query_ulang.csv"
)

statistik_acak.round(3).to_csv(
    hasil_dir / "statistik_query_acak.csv"
)

print("Statistik deskriptif berhasil disimpan.")

Statistik deskriptif berhasil disimpan.


### Interpretasi Statistik Deskriptif

Berdasarkan 300 observasi query ulang berstatus `ok`, Google Public DNS memiliki rata-rata waktu respons terendah sebesar **15,266 ms**, diikuti DNS bawaan jaringan sebesar **15,291 ms** dan Cloudflare sebesar **15,323 ms**. Perbedaan rata-rata antarketiga resolver relatif kecil.

Google juga memiliki standar deviasi terendah sebesar **15,018 ms**, dibandingkan DNS jaringan sebesar **17,367 ms** dan Cloudflare sebesar **18,751 ms**. Hal ini menunjukkan bahwa Google memiliki penyebaran waktu respons paling kecil pada pengukuran yang dilakukan.

Pada 300 observasi query acak, Google kembali memiliki rata-rata terendah sebesar **26,959 ms**, sedangkan DNS jaringan memiliki standar deviasi terendah sebesar **39,073 ms**.

Secara keseluruhan, rata-rata waktu respons query acak lebih tinggi daripada query ulang pada ketiga resolver. Selisih antara mean dan median menunjukkan adanya beberapa pengukuran dengan durasi tinggi yang memengaruhi rata-rata.

Hasil ini merupakan temuan deskriptif dari satu sesi pengukuran pada 24 November 2026. Perbedaan kinerja resolver belum dapat dinyatakan signifikan secara statistik sebelum dilakukan pengujian hipotesis.